In [16]:
q = 2

def key_size(m, n, w):
    return (m*m + n*w) // 8

def ct_size(m, n, w):
    return (2*m*n) // 8

def rGV_bound(q, n_code, k, m):
    """rGV для [n_code, k]_{q^m} кода."""
    w = 1
    while q_binomial(m, w, q) * q**(w*n_code) <= q**(m*(n_code - k)):
        w += 1
    return w - 1

def cpx_comb_RSD(q, n_code, k, m, w):
    """Комбинаторная атака [8] на IRSD, log2."""
    val = ((n_code - k) * m)**3 * q**(w * ceil((k+1)*m/n_code) - m)
    return int(log(max(1, val), 2))

def cpx_alg_RSD(q, n_code, k, m, w):
    """Алгебраическая атака [10] на IRSD, log2."""
    a = 0
    while m * binomial(n_code - k - 1, w) < binomial(n_code - a, w) - 1:
        a += 1
    val = q**(a*w) * m * binomial(n_code-k-1, w) * binomial(n_code-a, w)**2
    return int(log(max(1, val), 2))


def find_pir_params(L, q=2, threshold=143, max_m=30000):
    """
    Подбор минимальных параметров SHE для PIR.
    
    Фиксируем d=1 (одно умножение шифротекстов — достаточно для 2D PIR).
    
    Параметры:
      L         : число публикуемых шифротекстов клиента.
                  Для 2D PIR с N записями: L = 2*ceil(sqrt(N)).
      q         : размер базового поля (по умолчанию 2).
      threshold : порог сложности атак (143 ≈ 128 бит безопасности).
      max_m     : верхняя граница поиска по m.
    
    Возвращает (m, n, w) или None.
    """
    for m in range(L + 50, max_m):
        w_max_she = floor((sqrt(9 + 8*(m-1)) - 3) / 2)
        if w_max_she <= L:
            continue
        
        for n in range(max(20, L+1), m // 2 + 1):
            w_max_rgv = rGV_bound(q, 2*n, n, m)
            w_max = min(w_max_rgv, w_max_she)
            
            w_min = floor(L * n**2 / (n**2 + 1)) + 1
            
            if w_min > w_max:
                continue
            
            for w in range(w_min, w_max + 1):
                c1 = cpx_comb_RSD(q, n*(L+1), n, m, w)
                if c1 < threshold:
                    continue
                c2 = cpx_alg_RSD(q, n*(L+1), n, m, w)
                if c2 < threshold:
                    continue
                
                print("=" * 50)
                print(f"L = {L} публикуемых шифротекстов, q = {q}")
                print(f"  m = {m}")
                print(f"  n = {n}")
                print(f"  w = {w}")
                print("-" * 50)
                print(f"  Размер ключа:       {key_size(m, n, w)} B")
                print(f"  Размер шифротекста: {ct_size(m, n, w)} B")
                print(f"  Трафик от клиента:  {L * ct_size(m, n, w)} B")
                print(f"  cpx_comb = 2^{c1},  cpx_alg = 2^{c2}")
                print("=" * 50)
                return (m, n, w)
    
    print(f"Параметры для L={L} не найдены при m < {max_m}")
    return None


def pir_params(db_size, q=2, security=128):
    """
    Обертка для получения параметров PIR
    """
    L = 2 * ceil(sqrt(db_size))
    
    print(f"PIR конфигурация БД из {db_size} записей  =>  L = {L}")
    return find_pir_params(L, q=q, threshold=security+15)

In [17]:
find_pir_params(20)

L = 20 публикуемых шифротекстов, q = 2
  m = 351
  n = 28
  w = 25
--------------------------------------------------
  Размер ключа:       15487 B
  Размер шифротекста: 2457 B
  Трафик от клиента:  49140 B
  cpx_comb = 2^151,  cpx_alg = 2^443


(351, 28, 25)

In [18]:
pir_params(db_size=100)

PIR конфигурация БД из 100 записей  =>  L = 20
L = 20 публикуемых шифротекстов, q = 2
  m = 351
  n = 28
  w = 25
--------------------------------------------------
  Размер ключа:       15487 B
  Размер шифротекста: 2457 B
  Трафик от клиента:  49140 B
  cpx_comb = 2^151,  cpx_alg = 2^443


(351, 28, 25)

In [19]:
pir_params(db_size=10000)

PIR конфигурация БД из 10000 записей  =>  L = 200
L = 200 публикуемых шифротекстов, q = 2
  m = 20503
  n = 202
  w = 200
--------------------------------------------------
  Размер ключа:       52551676 B
  Размер шифротекста: 1035401 B
  Трафик от клиента:  207080200 B
  cpx_comb = 2^185,  cpx_alg = 2^5460


(20503, 202, 200)

In [20]:
def find_linear_pir_params(L, q=2, threshold=143):
    """
    Параметры AHE для линейного PIR с L шифротекстами в запросе.
    
    Используется только AHE (сложение + плейнтекст-умножение),
    поэтому условие SHE w(w+3)/2 + 1 < m НЕ применяется.
    
    Условия:
      1. w < m                          (корректность AHE)
      2. w ≤ rGV(q, 2n, n, m)           (единственность декодирования)
      3. L < w(1 + 1/n²)                (защита от линеаризации, Prop. 8)
      4. cpx_comb_RSD ≥ threshold       (комбинаторная атака на (L+1)-идеал. код)
      5. cpx_alg_RSD ≥ threshold        (алгебраическая атака на тот же код)
    
    Минимизируем размер шифротекста ct = 2mn/8.
    """
    L = int(L)
    q = int(q)
    best = None  # (ct_bytes, m, n, w, cpx_comb, cpx_alg)
    
    n_lo = max(L + 1, 20)
    n_hi = 4 * L + 50
    m_hi = 10 * L + 200
    
    for n in range(n_lo, n_hi + 1):
        # Для каждого n ищем минимальное m, удовлетворяющее всем условиям.
        # Минимальное m — то, при котором rGV >= w_min.
        w_min = int(L * n**2 // (n**2 + 1)) + 1
        
        for m in range(max(L + 2, n), m_hi + 1):
            w_max = min(int(rGV_bound(q, 2*n, n, m)), m - 1)
            if w_min > w_max:
                continue
            
            w = w_min
            c1 = cpx_comb_RSD(q, n*(L+1), n, m, w)
            if c1 < threshold:
                continue
            c2 = cpx_alg_RSD(q, n*(L+1), n, m, w)
            if c2 < threshold:
                continue
            
            # Нашли валидное (m, w) для этого n. Дальнейший рост m не уменьшит ct.
            ct = 2 * m * n // 8
            if best is None or ct < best[0]:
                best = (int(ct), int(m), int(n), int(w), int(c1), int(c2))
            break  # переходим к следующему n
    
    if best is None:
        print(f"Параметры для L={L} не найдены.")
        return None
    
    ct, m, n, w, c1, c2 = best
    key = (m*m + n*w) // 8
    
    # Явная конвертация Sage Rational/Integer в Python float для форматирования
    key_kb = float(key) / 1024.0
    ct_kb = float(ct) / 1024.0
    traffic_mb = float(L * ct) / 1024.0 / 1024.0
    
    print(f"=== Линейный PIR, L = N = {L}, q = {q} ===")
    print(f"  m = {m},  n = {n},  w = {w}")
    print(f"  ключ:        {key} B  ({key_kb:.1f} КБ)")
    print(f"  шифротекст:  {ct} B  ({ct_kb:.1f} КБ)")
    print(f"  трафик клиент→сервер ({L} ШТ):  {L*ct} B  ({traffic_mb:.2f} МБ)")
    print(f"  cpx_comb = 2^{c1},  cpx_alg = 2^{c2}")
    return (m, n, w)

In [21]:
find_linear_pir_params(100)

=== Линейный PIR, L = N = 100, q = 2 ===
  m = 322,  n = 182,  w = 100
  ключ:        15235 B  (14.9 КБ)
  шифротекст:  14651 B  (14.3 КБ)
  трафик клиент→сервер (100 ШТ):  1465100 B  (1.40 МБ)
  cpx_comb = 2^145,  cpx_alg = 2^2681


(322, 182, 100)

In [22]:
find_linear_pir_params(1000)

KeyboardInterrupt: 

In [ ]:
find_linear_pir_params(10)

In [ ]:
find_linear_pir_params(10000)

In [16]:
import time
from functools import lru_cache

def key_size(m, n, w):
    return (m * m + n * w) // 8

def ct_size(m, n, w):
    return (2 * m * n) // 8

def add_time(m, n):
    t = (2 * m * n) / 3000000
    return N(t) if t < 1 else floor(t)

def mul_time(m, n):
    t = (3 * (m * n) ** 1.6) / 3000000
    return N(t) if t < 1 else floor(t)

@lru_cache(maxsize=None)
def rGV_bound(q, n_code, k, m):
    """Ранговая граница Гилберта–Варшамова: максимально допустимый w."""
    w = 1
    rhs = q ** (m * (n_code - k))
    while q_binomial(m, w, q) * q ** (w * n_code) <= rhs:
        w += 1
    return w - 1

def w_she_max(m):
    from math import isqrt
    return int((isqrt(25 + 8*(m - 2)) - 5) // 2)

@lru_cache(maxsize=None)
def cpx_comb_RSD(q, n_code, k, m, w):
    """Комбинаторная атака [8] на IRSD, log2 сложности."""
    val = (((n_code - k) * m) ** 3) * (q ** (w * ceil((k + 1) * m / n_code) - m))
    return int(log(max(1, val), 2))

@lru_cache(maxsize=None)
def cpx_alg_RSD(q, n_code, k, m, w):
    """Алгебраическая атака [10] на IRSD, log2 сложности."""
    a = 0
    lhs = m * binomial(n_code - k - 1, w)
    while lhs < binomial(n_code - a, w) - 1:
        a += 1
    val = q ** (a * w) * m * binomial(n_code - k - 1, w) * (binomial(n_code - a, w) ** 2)
    return int(log(max(1, val), 2))

def _w_for(q, m, n, l, w_min, needs_she):
    """Минимальный валидный w для (m, n) либо None."""
    w_hi = rGV_bound(q, 2 * n, n, m)
    if needs_she:
        w_hi = min(w_hi, w_she_max(m))          
    if w_hi < w_min:
        return None
    w = w_min
    if l * n * n >= w * (n * n + 1):
        w = (l * n * n) // (n * n + 1) + 1
        if w > w_hi:
            return None
    return w

def _good(q, m, n, l, w_min, sec, needs_she):
    """Валидный w или None, с проверкой обеих атак на (l+1)-идеальный код."""
    w = _w_for(q, m, n, l, w_min, needs_she)
    if w is None:
        return None
    if cpx_comb_RSD(q, n * (l + 1), n, m, w) < sec:
        return None
    if cpx_alg_RSD(q, n * (l + 1), n, m, w) < sec:
        return None
    return w

def find_params_pir(N, pir_type, q=2, security_bits=128, safety_factor=None,
                    m_span=40000, n_max_cap=None, verbose=True):
    """Поиск (m, n, w). pir_type: 'obvious' | 'pir' | 'tensor'."""
    t0 = time.time()

    if pir_type == 'obvious':
        l, needs_she = N, False
    elif pir_type == 'pir':
        l, needs_she = int(ceil(sqrt(N))), False
    elif pir_type == 'tensor':
        l, needs_she = 2 * int(ceil(sqrt(N))), True
    else:
        raise ValueError("pir_type must be 'obvious' | 'pir' | 'tensor'")

    sf = Integer(4) / Integer(3) if safety_factor is None else QQ(safety_factor)
    w_min = int(ceil(sf * l))

    if needs_she:
        gens = w_min + w_min + w_min * (w_min + 1) // 2
        m_low = gens + 2 + max(10, w_min // 4)     
    else:
        m_low = w_min + 1
    n_min = max(40, w_min + 5)

    if verbose:
        scheme = 'SHE' if needs_she else 'AHE'
        print('===== PIR Parameter Search =====')
        print(f'  N (db size)      : {N}')
        print(f'  PIR type         : {pir_type}  ({scheme})')
        print(f'  ciphertexts (l)  : {l}')
        print(f'  required w >=    : {w_min}  (safety = {sf})')
        print(f'  m search from    : {m_low}')
        print(f'  security target  : {security_bits} bits')
        print()

    for m in range(m_low, m_low + m_span):
        n_hi = m // 2
        if n_max_cap is not None:
            n_hi = min(n_hi, n_max_cap)
        if n_hi < n_min:
            continue

        if needs_she:
            n = None
            for cand in range(n_min, min(n_hi, n_min + 400) + 1):
                if _good(q, m, cand, l, w_min, security_bits, needs_she) is not None:
                    n = cand
                    break
            if n is None:
                continue
        else:
            if _good(q, m, n_hi, l, w_min, security_bits, needs_she) is None:
                continue
            lo, hi, n = n_min, n_hi, None
            while lo <= hi:
                mid = (lo + hi) // 2
                if _good(q, m, mid, l, w_min, security_bits, needs_she) is not None:
                    n = mid
                    hi = mid - 1
                else:
                    lo = mid + 1

        w = _good(q, m, n, l, w_min, security_bits, needs_she)
        cc = cpx_comb_RSD(q, n * (l + 1), n, m, w)
        ca = cpx_alg_RSD(q, n * (l + 1), n, m, w)

        res = {
            'N': N, 'pir_type': pir_type, 'scheme': 'SHE' if needs_she else 'AHE',
            'q': q, 'l': l, 'm': m, 'n': n, 'w': w,
            'comb_security': cc, 'alg_security': ca, 'security': min(cc, ca),
            'key_size_B': key_size(m, n, w), 'ct_size_B': ct_size(m, n, w),
            'add_time_ms': add_time(m, n),
            'noise_ok': (not needs_she) or (w * (w + 3) // 2 + 1 < m),
            'search_time_s': time.time() - t0,
        }
        if needs_she:
            res['mul_time_ms'] = mul_time(m, n)

        if verbose:
            print(f'FOUND in {res["search_time_s"]:.1f}s')
            print(f'  m = {m},  n = {n},  w = {w}')
            print(f'  comb attack: {cc} bits   |   alg attack: {ca} bits')
            print(f'  key size: {res["key_size_B"]} B')
            print(f'  ct  size: {res["ct_size_B"]} B')
            print(f'  add: {res["add_time_ms"]} ms', end='')
            print(f'   mul_ct: {res["mul_time_ms"]} ms' if needs_she else '')
            chk = w * (w + 3) // 2 + 1
            print(f'  [SHE noise]  w(w+3)/2+1 = {chk}  <  m = {m}  ->  {chk < m}')
        return res

    if verbose:
        print(f'No suitable parameters found (m < {m_low + m_span}).')
    return None

In [17]:
find_params_pir(N=100, pir_type='obvious')   # AHE
# find_params_pir(N=1000, pir_type='obvious')  # AHE — уже тяжело
find_params_pir(N=100, pir_type='pir')     
find_params_pir(N=1000, pir_type='pir')     
find_params_pir(N=10000, pir_type='pir')     
find_params_pir(N=100, pir_type='tensor')    # SHE -> большое m
find_params_pir(N=1000, pir_type='tensor')    # SHE -> большое m
find_params_pir(N=10000, pir_type='tensor')    # SHE -> большое m

===== PIR Parameter Search =====
  N (db size)      : 100
  PIR type         : obvious  (AHE)
  ciphertexts (l)  : 100
  required w >=    : 134  (safety = 4/3)
  m search from    : 135
  security target  : 128 bits

FOUND in 2.7s
  m = 458,  n = 229,  w = 134
  comb attack: 281 bits   |   alg attack: 3558 bits
  key size: 30056 B
  ct  size: 26220 B
  add: 0.0699213333333333 ms
  [SHE noise]  w(w+3)/2+1 = 9180  <  m = 458  ->  False
===== PIR Parameter Search =====
  N (db size)      : 100
  PIR type         : pir  (AHE)
  ciphertexts (l)  : 10
  required w >=    : 14  (safety = 4/3)
  m search from    : 15
  security target  : 128 bits

FOUND in 0.2s
  m = 219,  n = 96,  w = 14
  comb attack: 128 bits   |   alg attack: 318 bits
  key size: 6163 B
  ct  size: 5256 B
  add: 0.0140160000000000 ms
  [SHE noise]  w(w+3)/2+1 = 120  <  m = 219  ->  True
===== PIR Parameter Search =====
  N (db size)      : 1000
  PIR type         : pir  (AHE)
  ciphertexts (l)  : 32
  required w >=    : 43  

{'N': 10000,
 'pir_type': 'tensor',
 'scheme': 'SHE',
 'q': 2,
 'l': 200,
 'm': 36380,
 'n': 272,
 'w': 267,
 'comb_security': 12306,
 'alg_security': 7299,
 'security': 7299,
 'key_size_B': 165447128,
 'ct_size_B': 2473840,
 'add_time_ms': 6,
 'noise_ok': True,
 'search_time_s': 23.68901300430298,
 'mul_time_ms': 155844}